# 01 — Data Exploration: Kaggle Indian Railways Delay Dataset

**Goal**: Understand data shape, completeness, and delay distribution to select 2–3 routes for backtesting.

Key questions:
1. What is the delay distribution? (median, mean, skew, heavy tails)
2. How much data is missing per column?
3. Which trains/routes have the most complete historical data?
4. Are there rake/train-linkage fields for the rake-inheritance feature?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Adjust path as needed after running the loader script
PROCESSED_DIR = Path("../data/processed")

# Try to load the cleaned parquet; fall back to listing available files
parquet_files = list(PROCESSED_DIR.glob("kaggle_*.parquet"))
if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    print(f"Loaded: {parquet_files[0].name}")
else:
    print("No cleaned Kaggle parquet found. Run src.ingestion.load_kaggle first.")
    print(f"Available in processed/: {list(PROCESSED_DIR.glob('*'))}")
    df = None

## 1. Basic Schema & Shape

In [ ]:
if df is not None:
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    df.head()

## 2. Missing Values Analysis

In [ ]:
if df is not None:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    missing_report = missing_report[missing_report.missing_count > 0].sort_values("missing_pct", ascending=False)
    
    if missing_report.empty:
        print("No missing values!")
    else:
        print("Columns with missing values:")
        print(missing_report)
        
        fig, ax = plt.subplots(figsize=(10, max(4, len(missing_report) * 0.4)))
        missing_report.missing_pct.plot.barh(ax=ax)
        ax.set_xlabel("% Missing")
        ax.set_title("Missing Values by Column")
        plt.tight_layout()
        plt.show()

## 3. Delay Distribution

In [ ]:
# Target column: 'actual_delay_minutes' — the only continuous delay target in the artifact.
# Earlier versions searched for alternative names ('delay_minutes', 'arrival_delay', etc.);
# those columns do not exist. This cell uses the correct name directly and raises KeyError if missing.
if df is not None:
    # The current artifact has one measured continuous delay target.
    delay_col = "actual_delay_minutes"
    if delay_col not in df.columns:
        raise KeyError(f"Expected {delay_col!r}; available columns: {list(df.columns)}")

    # Also report the separate binary target when present.
    binary_col = "delayed_gt_15min" if "delayed_gt_15min" in df.columns else None
    delays = pd.to_numeric(df[delay_col], errors="raise").dropna()
    print(f"Delay column: '{delay_col}'")
    print(f"  Count  : {len(delays):,}")
    print(f"  Mean   : {delays.mean():.2f}")
    print(f"  Median : {delays.median():.2f}")
    print(f"  Std    : {delays.std():.2f}")
    print(f"  P10    : {delays.quantile(0.1):.2f}")
    print(f"  P50    : {delays.quantile(0.5):.2f}")
    print(f"  P90    : {delays.quantile(0.9):.2f}")
    print(f"  Max    : {delays.max():.2f}")
    if binary_col:
        print(f"Binary target '{binary_col}' rate: {df[binary_col].mean() * 100:.1f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    clipped = delays.clip(upper=delays.quantile(0.99))
    axes[0].hist(clipped, bins=80, edgecolor='black', alpha=0.7)
    axes[0].axvline(delays.median(), color='red', linestyle='--', label=f'Median: {delays.median():.1f}')
    axes[0].axvline(delays.mean(), color='orange', linestyle='--', label=f'Mean: {delays.mean():.1f}')
    axes[0].set_xlabel(delay_col)
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Delay Distribution (clipped at P99)')
    axes[0].legend()
    axes[1].boxplot(clipped, vert=True)
    axes[1].set_ylabel(delay_col)
    axes[1].set_title('Delay Box Plot (clipped at P99)')
    plt.tight_layout()
    plt.show()

## 4. Train / Route Data Completeness

Identify which trains and routes have the most complete historical records — these are candidates for backtesting.

In [ ]:
if df is not None:
    train_col = "train_number"
    if train_col not in df.columns:
        raise KeyError(f"Expected {train_col!r}; available columns: {list(df.columns)}")

    train_counts = df[train_col].value_counts()
    print(f"Total unique trains: {df[train_col].nunique()}")
    print("\nTop 20 trains by record count:")
    top20 = train_counts.head(20)
    print(top20)

    fig, ax = plt.subplots(figsize=(12, 6))
    top20.plot.bar(ax=ax)
    ax.set_xlabel("Train Number")
    ax.set_ylabel("Number of Records")
    ax.set_title("Top 20 Trains by Data Volume")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    demo_trains = [12301, 12302, 12951, 12952, 12625, 12626]
    for train in demo_trains:
        count = int(train_counts.get(train, train_counts.get(str(train), 0)))
        print(f"  Train {train}: {count} records")

## 5. Rake / Rolling Stock Linkage Fields

The rake-inheritance feature needs a way to link a train's current run to its previous run using the same physical rake. Look for columns like `rake_id`, `coach_composition`, `loco_number`, `consist_id`, etc.

In [ ]:
if df is not None:
    rake_keywords = ["rake", "rolling", "stock", "consist", "composition", "engine", "wagon"]
    rake_cols = [c for c in df.columns if any(kw in c.lower() for kw in rake_keywords)]
    proxy_cols = [c for c in df.columns if c in {"coach_count", "loco_age_years"}]

    if rake_cols:
        print("Potential explicit rake/rolling-stock fields:", rake_cols)
    else:
        print("No physical rake identifier found in the current artifact.")
    if proxy_cols:
        print("Non-identifying rolling-stock proxies:", proxy_cols)
    print("The current feature uses prior delay for the same train number, not a physical rake ID.")

## 6. Summary & Route Selection

Based on the analysis above, we select routes for backtesting based on:
- Data completeness (number of historical records)
- Route diversity (long-distance, different zones)
- RailRadar crowd-sourced GPS reliability (popular routes = better coverage)

**Selected routes** (pending confirmation from data above):
1. **12301/12302 Howrah Rajdhani** (NDLS ↔ HWH) — extremely popular, good coverage
2. **12951/12952 Mumbai Rajdhani** (NDLS ↔ BCT) — parallel DFC, good for testing capacity effects
3. **12625/12626 Kerala Express** (NDLS ↔ TVC) — long route, high delay variability

In [ ]:
print("Notebook complete. Proceed to feature engineering after confirming route selection.")